In [ ]:
import os
import glob
import shutil
import pandas as pd
from datetime import datetime, timedelta

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------
VISIT_GAP_THRESHOLD_SECONDS = 3
GAP_THRESHOLD = timedelta(seconds=VISIT_GAP_THRESHOLD_SECONDS)
MINIMUM_VISIT_DURATION = timedelta(seconds=30)
BUFFER = timedelta(minutes=0.1)  # strict mode -> no buffer

OUTPUT_ROOT = r"D:\CVPR_Data\Matched_SOW_ID_Visit_Frames\Day1_2"  # <-- where to copy frames

CAMERA_MAPPING = {
    "Camera1": "2C",
    "Camera2": "2A",
    "Camera3": "2B"
}

CAMERA_DIRS = {
    "Camera1": r"D:\Camera1_transfer_11_04_2025",
    "Camera2": r"D:\Camera2_transfer_11_04_2025",
    "Camera3": r"D:\Camera3_transfer_11_04_2025"
}

ACTUAL_TIME = "2025-10-31 11:47:00"
CAMERA_TIME = "1969-12-31 18:04:00"

CSV_FILE = r"C:\Users\spaudel\Downloads\Sow Export 10_31_11_4.xlsx"


# =========================================================
# LOAD FEEDER LOG
# =========================================================
def load_and_filter_feeder_data(filepath, feeder_id):
    df = pd.read_excel(filepath) if filepath.endswith(("xls", "xlsx")) else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()

    df["Start"] = pd.to_datetime(df["Start"])
    df["End"]   = pd.to_datetime(df["End"])
    df["FEEDER_STR"] = df["FEEDER"].astype(str).str.strip()

    return df[df["FEEDER_STR"] == str(feeder_id)]


# =========================================================
# CONVERT CAMERA FILENAME → REAL TIME
# =========================================================
def get_real_capture_details(filename, time_offset, root):
    parts = filename.split("_")
    if len(parts) < 5:
        return None

    date_str = parts[1]
    time_str = parts[2].replace("-", ":")
    frame_num = parts[3]

    timestamp = f"{date_str} {time_str}"

    try:
        recorded = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        real_time = recorded + time_offset
        return {
            "filename": filename,
            "real_time": real_time,
            "frame_number": int(frame_num),
            "filepath": os.path.join(root, filename),
        }
    except:
        return None


# =========================================================
# GROUP FRAMES INTO VISITS
# =========================================================
def extract_camera_visits(directory, actual_time, camera_time):
    actual = datetime.strptime(actual_time, "%Y-%m-%d %H:%M:%S")
    cam = datetime.strptime(camera_time, "%Y-%m-%d %H:%M:%S")

    time_offset = actual - cam

    files = glob.glob(os.path.join(directory, "*_depth.png"))
    parsed = []

    for f in files:
        d = get_real_capture_details(os.path.basename(f), time_offset, directory)
        if d:
            parsed.append(d)

    parsed.sort(key=lambda x: x["real_time"])

    # Group frames
    raw_visits = []
    current = []

    for i, f in enumerate(parsed):
        if i == 0:
            current.append(f)
            continue

        gap = f["real_time"] - parsed[i - 1]["real_time"]
        if gap > GAP_THRESHOLD:
            raw_visits.append(current)
            current = [f]
        else:
            current.append(f)

    if current:
        raw_visits.append(current)

    # Filter short visits
    long_visits = [
        v for v in raw_visits
        if (v[-1]["real_time"] - v[0]["real_time"]) >= MINIMUM_VISIT_DURATION
    ]

    return long_visits, parsed


# =========================================================
# STRICT MATCHING: visit must be INSIDE feeder visit
# =========================================================
def match_visits_to_sows(visits, feeder_df):
    results = []
    for i, v in enumerate(visits, start=1):
        cam_start = v[0]["real_time"]
        cam_end   = v[-1]["real_time"]

        matches = feeder_df[
            (feeder_df["Start"] <= cam_start) &
            (feeder_df["End"]   >= cam_end)
        ]

        sow_ids = matches["SOW"].unique().tolist() if len(matches) > 0 else []

        results.append({
            "VisitID": i,
            "CameraStart": cam_start,
            "CameraEnd": cam_end,
            "Frames": len(v),
            "SOW_IDs": sow_ids,
            "FramesList": v  # SAVE actual frames for copying
        })

    return results


# =========================================================
# COPY FRAMES FOR EXACTLY ONE MATCHED SOW
# =========================================================
def copy_single_sow_frames(cam_name, matches):
    copied = 0
    skipped_zero = 0
    skipped_multi = 0

    for m in matches:
        sow_ids = m["SOW_IDs"]

        if len(sow_ids) == 0:
            skipped_zero += 1
            continue

        if len(sow_ids) > 1:
            skipped_multi += 1
            continue

        sow_id = str(sow_ids[0])

        # Make consistent visit folder
        start_t = m["CameraStart"].strftime("%Y-%m-%d_%H-%M-%S")
        end_t   = m["CameraEnd"].strftime("%Y-%m-%d_%H-%M-%S")

        visit_folder = os.path.join(OUTPUT_ROOT, sow_id, cam_name, f"{start_t}__{end_t}")
        os.makedirs(visit_folder, exist_ok=True)

        # Copy frames
        for frame in m["FramesList"]:
            src = frame["filepath"]
            dst = os.path.join(visit_folder, frame["filename"])
            shutil.copy2(src, dst)
            copied += 1

    print(f"\n=== COPY SUMMARY FOR {cam_name} ===")
    print(f"Copied frames: {copied}")
    print(f"Skipped (0-ID visits): {skipped_zero}")
    print(f"Skipped (multi-ID visits): {skipped_multi}")
    print("====================================\n")

# =========================================================
# MASTER LOOP
# =========================================================
if __name__ == "__main__":

    START_CUTOFF = datetime(2025, 10, 31, 11, 47, 0)
    END_CUTOFF   = START_CUTOFF + timedelta(hours=24)
    for cam_name, feeder_id in CAMERA_MAPPING.items():
        print("\n============================================")
        print(f"PROCESSING {cam_name} (Feeder {feeder_id})")
        print("============================================")

        feeder_df = load_and_filter_feeder_data(CSV_FILE, feeder_id)

        visits, all_frames = extract_camera_visits(
            CAMERA_DIRS[cam_name],
            ACTUAL_TIME,
            CAMERA_TIME
        )

        # ⭐ Filter visits BEFORE matching or copying
        visits = [
            v for v in visits
            if (v[0]["real_time"] >= START_CUTOFF and v[-1]["real_time"] <= END_CUTOFF)
        ]

        # Match filtered visits only
        matches = match_visits_to_sows(visits, feeder_df)

        # Copy frames for accepted visits
        copy_single_sow_frames(cam_name, matches)





PROCESSING Camera1 (Feeder 2C)

=== COPY SUMMARY FOR Camera1 ===
Copied frames: 2498
Skipped (0-ID visits): 0
Skipped (multi-ID visits): 179


PROCESSING Camera2 (Feeder 2A)

=== COPY SUMMARY FOR Camera2 ===
Copied frames: 4998
Skipped (0-ID visits): 14
Skipped (multi-ID visits): 169


PROCESSING Camera3 (Feeder 2B)

=== COPY SUMMARY FOR Camera3 ===
Copied frames: 10309
Skipped (0-ID visits): 53
Skipped (multi-ID visits): 0



Following code will run for only 10 stbale pigs

In [12]:
import os
import glob
import shutil
import pandas as pd
from datetime import datetime, timedelta

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------
VISIT_GAP_THRESHOLD_SECONDS = 3
GAP_THRESHOLD = timedelta(seconds=VISIT_GAP_THRESHOLD_SECONDS)
MINIMUM_VISIT_DURATION = timedelta(seconds=30)
BUFFER = timedelta(minutes=0.1)  # strict mode -> no buffer

OUTPUT_ROOT = r"D:\CVPR_Data\Matched_SOW_ID_Visit_Frames\Day4_2"  # <-- where to copy frames

CAMERA_MAPPING = {
    "Camera1": "2C",
    "Camera2": "2A",
    "Camera3": "2B"
}

CAMERA_DIRS = {
    "Camera1": r"D:\Camera1_transfer_11_04_2025",
    "Camera2": r"D:\Camera2_transfer_11_04_2025",
    "Camera3": r"D:\Camera3_transfer_11_04_2025"
}

ACTUAL_TIME = "2025-10-31 11:47:00"
CAMERA_TIME = "1969-12-31 18:04:00"

CSV_FILE = r"C:\Users\spaudel\Downloads\Sow Export 10_31_11_4.xlsx"

# ✅ Only keep these 10 sows
STABLE_LIST = ['419', '453', '458', '467', '472', '485', '492', '495', '497', '542-533']
STABLE_SET = set(str(s).strip() for s in STABLE_LIST)

# =========================================================
# LOAD FEEDER LOG
# =========================================================
def load_and_filter_feeder_data(filepath, feeder_id):
    df = pd.read_excel(filepath) if filepath.endswith(("xls", "xlsx")) else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()

    df["Start"] = pd.to_datetime(df["Start"])
    df["End"]   = pd.to_datetime(df["End"])
    df["FEEDER_STR"] = df["FEEDER"].astype(str).str.strip()
    df["SOW_STR"] = df["SOW"].astype(str).str.strip()

    # Filter to feeder + stable sows only
    df = df[(df["FEEDER_STR"] == str(feeder_id)) & (df["SOW_STR"].isin(STABLE_SET))]
    return df


# =========================================================
# CONVERT CAMERA FILENAME → REAL TIME
# =========================================================
def get_real_capture_details(filename, time_offset, root):
    parts = filename.split("_")
    if len(parts) < 5:
        return None

    date_str = parts[1]
    time_str = parts[2].replace("-", ":")
    frame_num = parts[3]

    timestamp = f"{date_str} {time_str}"

    try:
        recorded = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        real_time = recorded + time_offset
        return {
            "filename": filename,
            "real_time": real_time,
            "frame_number": int(frame_num),
            "filepath": os.path.join(root, filename),
        }
    except:
        return None


# =========================================================
# GROUP FRAMES INTO VISITS
# =========================================================
def extract_camera_visits(directory, actual_time, camera_time):
    actual = datetime.strptime(actual_time, "%Y-%m-%d %H:%M:%S")
    cam = datetime.strptime(camera_time, "%Y-%m-%d %H:%M:%S")

    time_offset = actual - cam

    files = glob.glob(os.path.join(directory, "*_depth.png"))
    parsed = []

    for f in files:
        d = get_real_capture_details(os.path.basename(f), time_offset, directory)
        if d:
            parsed.append(d)

    parsed.sort(key=lambda x: x["real_time"])

    # Group frames
    raw_visits = []
    current = []

    for i, f in enumerate(parsed):
        if i == 0:
            current.append(f)
            continue

        gap = f["real_time"] - parsed[i - 1]["real_time"]
        if gap > GAP_THRESHOLD:
            raw_visits.append(current)
            current = [f]
        else:
            current.append(f)

    if current:
        raw_visits.append(current)

    # Filter short visits
    long_visits = [
        v for v in raw_visits
        if (v[-1]["real_time"] - v[0]["real_time"]) >= MINIMUM_VISIT_DURATION
    ]

    return long_visits, parsed


# =========================================================
# STRICT MATCHING: visit must be INSIDE feeder visit
# and exactly ONE stable sow must match
# =========================================================
def match_visits_to_sows(visits, feeder_df):
    results = []
    for i, v in enumerate(visits, start=1):
        cam_start = v[0]["real_time"]
        cam_end   = v[-1]["real_time"]

        matches = feeder_df[
            (feeder_df["Start"] <= cam_start) &
            (feeder_df["End"]   >= cam_end)
        ]

        # only stable sow IDs (already filtered, but keep safe)
        sow_ids = matches["SOW_STR"].unique().tolist() if len(matches) > 0 else []

        results.append({
            "VisitID": i,
            "CameraStart": cam_start,
            "CameraEnd": cam_end,
            "Frames": len(v),
            "SOW_IDs": sow_ids,
            "FramesList": v
        })

    return results


# =========================================================
# COPY FRAMES FOR EXACTLY ONE MATCHED STABLE SOW
# =========================================================
def copy_single_sow_frames(cam_name, matches):
    copied = 0
    skipped_zero = 0
    skipped_multi = 0
    skipped_not_stable = 0

    for m in matches:
        sow_ids = m["SOW_IDs"]

        if len(sow_ids) == 0:
            skipped_zero += 1
            continue

        if len(sow_ids) > 1:
            skipped_multi += 1
            continue

        sow_id = str(sow_ids[0]).strip()
        if sow_id not in STABLE_SET:
            skipped_not_stable += 1
            continue

        # Make consistent visit folder
        start_t = m["CameraStart"].strftime("%Y-%m-%d_%H-%M-%S")
        end_t   = m["CameraEnd"].strftime("%Y-%m-%d_%H-%M-%S")

        visit_folder = os.path.join(OUTPUT_ROOT, sow_id, cam_name, f"{start_t}__{end_t}")
        os.makedirs(visit_folder, exist_ok=True)

        # Copy frames
        for frame in m["FramesList"]:
            src = frame["filepath"]
            dst = os.path.join(visit_folder, frame["filename"])
            shutil.copy2(src, dst)
            copied += 1

    print(f"\n=== COPY SUMMARY FOR {cam_name} (stable 10 only) ===")
    print(f"Copied frames: {copied}")
    print(f"Skipped (0-ID visits): {skipped_zero}")
    print(f"Skipped (multi-ID visits): {skipped_multi}")
    print(f"Skipped (not in stable list): {skipped_not_stable}")
    print("====================================\n")


# =========================================================
# MASTER LOOP
# =========================================================
if __name__ == "__main__":

    START_CUTOFF = datetime(2025, 10, 31, 11, 47, 0) + timedelta(hours=72)
    END_CUTOFF   = START_CUTOFF + timedelta(hours=24)

    print("Using STABLE_LIST (10 sows):", STABLE_LIST)

    for cam_name, feeder_id in CAMERA_MAPPING.items():
        print("\n============================================")
        print(f"PROCESSING {cam_name} (Feeder {feeder_id})")
        print("============================================")

        feeder_df = load_and_filter_feeder_data(CSV_FILE, feeder_id)

        visits, all_frames = extract_camera_visits(
            CAMERA_DIRS[cam_name],
            ACTUAL_TIME,
            CAMERA_TIME
        )

        # Filter visits BEFORE matching or copying
        visits = [
            v for v in visits
            if (v[0]["real_time"] >= START_CUTOFF and v[-1]["real_time"] <= END_CUTOFF)
        ]

        # Match filtered visits only
        matches = match_visits_to_sows(visits, feeder_df)

        # Copy frames for accepted visits (exactly one stable sow)
        copy_single_sow_frames(cam_name, matches)


Using STABLE_LIST (10 sows): ['419', '453', '458', '467', '472', '485', '492', '495', '497', '542-533']

PROCESSING Camera1 (Feeder 2C)

=== COPY SUMMARY FOR Camera1 (stable 10 only) ===
Copied frames: 5110
Skipped (0-ID visits): 127
Skipped (multi-ID visits): 0
Skipped (not in stable list): 0


PROCESSING Camera2 (Feeder 2A)

=== COPY SUMMARY FOR Camera2 (stable 10 only) ===
Copied frames: 3789
Skipped (0-ID visits): 215
Skipped (multi-ID visits): 0
Skipped (not in stable list): 0


PROCESSING Camera3 (Feeder 2B)

=== COPY SUMMARY FOR Camera3 (stable 10 only) ===
Copied frames: 4764
Skipped (0-ID visits): 252
Skipped (multi-ID visits): 0
Skipped (not in stable list): 0

